In [3]:
import os
import cv2
import pytesseract
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array  # Tambahkan import ini
from tensorflow.keras.preprocessing import image
# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk melakukan OCR pada gambar KTP
def ocr_on_ktp(image):
    # Mengubah gambar menjadi grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Preprocessing gambar untuk meningkatkan akurasi OCR
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # Menggunakan Tesseract untuk mengekstrak teks
    text = pytesseract.image_to_string(thresh, lang='eng')

    return text


# Fungsi untuk memuat gambar dan memprediksi label menggunakan CNN model
def predict_with_model(image, model):
    # Mengubah gambar menjadi format yang sesuai untuk prediksi
    img_resized = cv2.resize(image, (224, 224))
    img_array = img_to_array(img_resized)  # Gunakan fungsi img_to_array dari Keras
    img_array = np.expand_dims(img_array, axis=0) / 255.0  # Normalisasi

    # Melakukan prediksi dengan model
    pred = model.predict(img_array)

    # Menampilkan prediksi
    if pred >= 0.5:
        return "Class 1"  # KTP dengan kelas 1
    else:
        return "Class 0"  # KTP dengan kelas 0

# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    # Dapatkan semua file .h5 dari folder model_save_path
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    # Pilih model yang terakhir (berdasarkan urutan nama file atau buat logika untuk memilih model)
    model_file = sorted(model_files)[-1]  # Model yang terakhir disimpan
    model_path = os.path.join(model_save_path, model_file)

    # Memuat model
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model


# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)  # Menggunakan kamera HP (biasanya indeks kamera eksternal adalah 1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam! Pastikan kamera terhubung dan tidak digunakan oleh aplikasi lain.")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    # Membaca frame dari kamera DroidCam
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame! Pastikan kamera DroidCam berfungsi dengan baik.")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        # Menunggu input dari pengguna untuk melakukan scanning OCR
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            predicted_class = predict_with_model(frame, model)

            # Menampilkan hasil OCR dan prediksi
            print(f"Teks yang terdeteksi oleh OCR: {ocr_text}")
            print(f"Prediksi untuk gambar KTP: {predicted_class}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"  # Path lokal untuk menyimpan gambar
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()


Model yang dimuat: model_fold_4.h5
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 582ms/step
Teks yang terdeteksi oleh OCR: PROVINSI JAWA TIMUR
KABUPATEN SIDOARJO

' 3b031850%e040015


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
Teks yang terdeteksi oleh OCR: PROVINS! JAWA TIMUR
KABUPATEN SIDOARJO

Ik + 3bO31L850b20490015

AISHKA SYAKIRAH WIBOWO
‘eV Top ahi: MALANG, 19-12-2006
kelowin — : PEREMPUAN


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
